# Generating segmentation predictions

As the upcoming model is used to refine segmentations... # TODO

In [ ]:
import torch
import os
import pytorch_lightning as pl

from image_segmentation.data import ImageDataset
from image_segmentation.models import BinarySegmentator

## Loading the model

### Case #1: Use the provided weights (see [README](../README.md)) to initialize the pretrained U-Net model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

provided_checkpoint_dir = os.path.abspath('../checkpoints/unet_pretrained')
provided_ckpt_path = os.path.join(provided_checkpoint_dir, os.listdir(provided_checkpoint_dir)[0])

model = BinarySegmentator.load_from_checkpoint(provided_ckpt_path, map_location=device)

### Case #2: Use the U-Net model trained in the [preceding notebook (2)](./02_pretrain_unet.ipynb)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint_dir = os.path.abspath('../checkpoints/unet_pretraining')
ckpt_path = os.path.join(checkpoint_dir, os.listdir(checkpoint_dir)[0])

model = BinarySegmentator.load_from_checkpoint(ckpt_path, map_location=device)



### Case #3: Use your own model. In this case you will need to adapt the code below to load your model and its weights, and to use it for generating the binary predictions.

### Case #4: You already have precomputed predictions, or want to use the path classification model on ground truths, you don't need to execute the following cells of this notebook and can directly go to the [path train data generation notebook (4)](./04_path_train_data_generation.ipynb)

## Generating the prediction masks

In [ ]:
import os

data_dir = os.path.abspath('../data/FIVES')
dataset = ImageDataset(data_dir, transforms=None)

In [ ]:
import matplotlib.pyplot as plt
import gc

pred_dir = os.path.join(data_dir, 'pred')
os.makedirs(pred_dir, exist_ok=True)

for i, img_path in enumerate(dataset.img_list):
    img = dataset[i][0].transpose(2, 0, 1)
    img_tensor = torch.tensor(img).unsqueeze(0).float().to(device)
    with torch.no_grad():
        pred = model(img_tensor).squeeze().cpu().numpy()
        pred_bin = (pred > 0.5).astype('uint8') * 255
    pred_path = os.path.join(pred_dir, os.path.basename(img_path))
    plt.imsave(pred_path, pred_bin, cmap='gray')
    del img_tensor, pred, pred_bin, img

gc.collect()
torch.cuda.empty_cache()

In [ ]:
print(f"Predictions saved to {pred_dir}, total {len(os.listdir(pred_dir))} images.")

Now that we have prediction masks to execute the main model on, you can continue on the [path train data generation notebook (4)](./04_path_train_data_generation.ipynb)